# RT Notebook 26 — Self-Contained D/E Comparison

Run this notebook immediately in Google Colab, JupyterLab, or local Jupyter.

It has no external project dependencies. By default it uses an embedded reference candidate, generates its own frozen corpus, compares candidate and clean-room outputs, performs shuffled replay, preserves disagreements, writes evidence files, calculates hashes, and creates a ZIP archive.

Leave `USE_EXTERNAL_CANDIDATE = False` for the first run. Later, paste a governed candidate into the marked functions and change it to `True`.

The default run is harness validation only, with evidence ceiling `C1_EXECUTION_RECONSTRUCTION`.


In [ ]:
from pathlib import Path
import json, hashlib, random, copy, zipfile, platform, sys
from datetime import datetime, timezone

ROOT = Path.cwd()
NOTEBOOK_ID = "RT_NOTEBOOK_26_D_E_SELF_CONTAINED_001"
RESULT_ID = NOTEBOOK_ID + "_RESULTS_001"
OUT = ROOT / RESULT_ID
OUT.mkdir(parents=True, exist_ok=True)

def canonical(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False)

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path):
    return sha256_bytes(Path(path).read_bytes())

def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False)+"\n", encoding="utf-8")

def write_jsonl(path, rows):
    with Path(path).open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(canonical(row)+"\n")

USE_EXTERNAL_CANDIDATE = False
RUN_MODE = "EXTERNAL_CANDIDATE" if USE_EXTERNAL_CANDIDATE else "REFERENCE_STANDIN"
THRESHOLD_ENV = {"alpha":0.125,"beta":0.375,"gamma":0.625,"delta":0.875}
CONTEXTS=["K1","K2","K3","K4"]
SEEDS=[2601,2609,2617,2621]
write_json(OUT/"threshold_environment.json", THRESHOLD_ENV)
print("Run mode:", RUN_MODE)
print("Output:", OUT.resolve())


In [ ]:
def cr_same_context(a,b):
    return isinstance(a,str) and isinstance(b,str) and bool(a) and a==b

def cr_exact_bind(witness,payload):
    return isinstance(witness,dict) and witness.get("token")==payload

def cr_ordered_history(history):
    if not isinstance(history,list) or not history:
        return False
    if not all(isinstance(x,dict) and "step" in x and "state" in x for x in history):
        return False
    steps=[x["step"] for x in history]
    return all(isinstance(s,int) and not isinstance(s,bool) for s in steps) and all(steps[i] < steps[i+1] for i in range(len(steps)-1))

def cleanroom_representable(r,env):
    checks=[
        ("REJECT_TYPE", r.get("relation_type")=="SourceRelation"),
        ("REJECT_CONTEXT", cr_same_context(r.get("context"),r.get("target_context"))),
        ("REJECT_PROFILE", r.get("profile") in env),
        ("REJECT_WITNESS", isinstance(r.get("witness"),dict)),
        ("REJECT_WITNESS", cr_exact_bind(r.get("witness"),r.get("source_payload"))),
        ("REJECT_HISTORY", isinstance(r.get("history"),list) and bool(r.get("history"))),
        ("REJECT_HISTORY", cr_ordered_history(r.get("history"))),
        ("REJECT_HISTORY", isinstance(r.get("history"),list) and bool(r.get("history")) and r["history"][-1].get("state")==r.get("target")),
    ]
    for code,passed in checks:
        if not passed:
            return code
    return "REPRESENTABLE"

def cleanroom_noncollapsed(r,env):
    p=r.get("profile")
    if p not in env:
        return "REJECT_PROFILE"
    d=r.get("distinction")
    if not isinstance(d,(int,float)) or isinstance(d,bool) or d<=0:
        return "REJECT_DISTINCTION"
    if d<=env[p]:
        return "REJECT_SUBTHRESHOLD"
    return "NON_COLLAPSED"

def cleanroom_admissible(r,env):
    return cleanroom_representable(r,env)=="REPRESENTABLE" and cleanroom_noncollapsed(r,env)=="NON_COLLAPSED"


In [ ]:
# OPTIONAL EXTERNAL CANDIDATE: replace these three functions only when ready.
def external_candidate_representable(record, environment):
    raise NotImplementedError("Paste governed candidate representability here")

def external_candidate_noncollapsed(record, environment):
    raise NotImplementedError("Paste governed candidate non-collapse here")

def external_candidate_admissible(record, environment):
    raise NotImplementedError("Paste governed candidate admissibility here")

# Embedded reference candidate for immediate execution.
def reference_candidate_representable(r,env):
    if r.get("relation_type")!="SourceRelation": return "REJECT_TYPE"
    c,t=r.get("context"),r.get("target_context")
    if not isinstance(c,str) or not isinstance(t,str) or not c or c!=t: return "REJECT_CONTEXT"
    if r.get("profile") not in env: return "REJECT_PROFILE"
    w=r.get("witness")
    if not isinstance(w,dict) or w.get("token")!=r.get("source_payload"): return "REJECT_WITNESS"
    h=r.get("history")
    if not isinstance(h,list) or not h: return "REJECT_HISTORY"
    if not all(isinstance(x,dict) and "step" in x and "state" in x for x in h): return "REJECT_HISTORY"
    steps=[x["step"] for x in h]
    if not all(isinstance(s,int) and not isinstance(s,bool) for s in steps): return "REJECT_HISTORY"
    if any(steps[i]>=steps[i+1] for i in range(len(steps)-1)): return "REJECT_HISTORY"
    if h[-1].get("state")!=r.get("target"): return "REJECT_HISTORY"
    return "REPRESENTABLE"

def reference_candidate_noncollapsed(r,env):
    p=r.get("profile")
    if p not in env: return "REJECT_PROFILE"
    d=r.get("distinction")
    if not isinstance(d,(int,float)) or isinstance(d,bool) or d<=0: return "REJECT_DISTINCTION"
    if d<=env[p]: return "REJECT_SUBTHRESHOLD"
    return "NON_COLLAPSED"

def reference_candidate_admissible(r,env):
    return reference_candidate_representable(r,env)=="REPRESENTABLE" and reference_candidate_noncollapsed(r,env)=="NON_COLLAPSED"

if USE_EXTERNAL_CANDIDATE:
    candidate_representable=external_candidate_representable
    candidate_noncollapsed=external_candidate_noncollapsed
    candidate_admissible=external_candidate_admissible
    CANDIDATE_ID="USER_SUPPLIED_EXTERNAL_CANDIDATE"
else:
    candidate_representable=reference_candidate_representable
    candidate_noncollapsed=reference_candidate_noncollapsed
    candidate_admissible=reference_candidate_admissible
    CANDIDATE_ID="SELF_CONTAINED_REFERENCE_STANDIN"
print("Candidate:", CANDIDATE_ID)


In [ ]:
def make_baseline(seed,context,profile,ordinal):
    rng=random.Random(f"N26:{seed}:{context}:{profile}:{ordinal}")
    payload={"left":rng.randint(1,1000),"right":[rng.randint(1,50),rng.randint(51,100)]}
    target=f"TARGET_{context}_{seed}_{ordinal}"
    threshold=THRESHOLD_ENV[profile]
    return {"row_id":f"N26_B_{seed}_{context}_{profile}_{ordinal}","relation_type":"SourceRelation","context":context,"target_context":context,"source_payload":payload,"witness":{"token":copy.deepcopy(payload),"review":"self-contained"},"history":[{"step":10,"state":f"START_{ordinal}"},{"step":20,"state":f"MID_{ordinal}"},{"step":30,"state":target}],"target":target,"profile":profile,"distinction":round(threshold+0.01+rng.random()*0.05,9),"metadata":{"seed":seed,"ordinal":ordinal},"family":"baseline"}

BASELINES=[make_baseline(s,c,p,o) for s in SEEDS for c in CONTEXTS for p in THRESHOLD_ENV for o in range(2)]
FAULTS=["wrong_type","empty_context","cross_context","unknown_profile","missing_witness","witness_scalar","witness_corruption","missing_history","empty_history","malformed_history","boolean_history_step","duplicate_history_step","descending_history","false_terminal","nonnumeric_distinction","boolean_distinction","zero_distinction","negative_distinction","threshold_minus_epsilon","threshold_exact","threshold_plus_epsilon","metadata_enrichment","combined_type_context_fault","combined_context_profile_fault","combined_all_faults"]

def mutate(row,f):
    r=copy.deepcopy(row); r["row_id"]=f"{row['row_id']}__{f}"; r["parent_id"]=row["row_id"]; r["family"]=f
    if f=="wrong_type": r["relation_type"]="OtherRelation"
    elif f=="empty_context": r["context"]=""; r["target_context"]=""
    elif f=="cross_context": r["target_context"]=next(x for x in CONTEXTS if x!=r["context"])
    elif f=="unknown_profile": r["profile"]="omega"
    elif f=="missing_witness": r["witness"]=None
    elif f=="witness_scalar": r["witness"]=17
    elif f=="witness_corruption": r["witness"]["token"]["left"]+=1
    elif f=="missing_history": r.pop("history",None)
    elif f=="empty_history": r["history"]=[]
    elif f=="malformed_history": r["history"][1]={"step":20}
    elif f=="boolean_history_step": r["history"][1]["step"]=True
    elif f=="duplicate_history_step": r["history"][1]["step"]=r["history"][0]["step"]
    elif f=="descending_history": r["history"]=list(reversed(r["history"]))
    elif f=="false_terminal": r["history"][-1]["state"]="WRONG_TARGET"
    elif f=="nonnumeric_distinction": r["distinction"]="0.9"
    elif f=="boolean_distinction": r["distinction"]=True
    elif f=="zero_distinction": r["distinction"]=0
    elif f=="negative_distinction": r["distinction"]=-0.5
    elif f=="threshold_minus_epsilon": r["distinction"]=THRESHOLD_ENV[r["profile"]]-1e-9
    elif f=="threshold_exact": r["distinction"]=THRESHOLD_ENV[r["profile"]]
    elif f=="threshold_plus_epsilon": r["distinction"]=THRESHOLD_ENV[r["profile"]]+1e-9
    elif f=="metadata_enrichment": r["metadata"]["extra"]={"nested":[1,2,3],"note":"nonsemantic"}
    elif f=="combined_type_context_fault": r["relation_type"]="OtherRelation"; r["target_context"]="BAD_CONTEXT"
    elif f=="combined_context_profile_fault": r["target_context"]="BAD_CONTEXT"; r["profile"]="omega"
    elif f=="combined_all_faults": r["relation_type"]="OtherRelation"; r["target_context"]="BAD_CONTEXT"; r["profile"]="omega"; r["witness"]=None; r["history"]=[]; r["distinction"]=-1
    return r

ROWS=[]
for b in BASELINES:
    ROWS.append(copy.deepcopy(b)); ROWS.extend(mutate(b,f) for f in FAULTS)
CORPUS_SHA256=sha256_bytes(canonical(ROWS).encode())
write_jsonl(OUT/"comparison_corpus.jsonl",ROWS)
write_json(OUT/"corpus_manifest.json",{"baselines":len(BASELINES),"fault_families":len(FAULTS),"rows_per_pass":len(ROWS),"corpus_sha256":CORPUS_SHA256,"contexts":CONTEXTS,"seeds":SEEDS})
print("Baselines:",len(BASELINES),"Fault families:",len(FAULTS),"Rows per pass:",len(ROWS))


In [ ]:
def eval_o(r):
    return {"row_id":r["row_id"],"representable":cleanroom_representable(copy.deepcopy(r),copy.deepcopy(THRESHOLD_ENV)),"noncollapsed":cleanroom_noncollapsed(copy.deepcopy(r),copy.deepcopy(THRESHOLD_ENV)),"admissible":cleanroom_admissible(copy.deepcopy(r),copy.deepcopy(THRESHOLD_ENV))}

def eval_c(r):
    return {"row_id":r["row_id"],"representable":candidate_representable(copy.deepcopy(r),copy.deepcopy(THRESHOLD_ENV)),"noncollapsed":candidate_noncollapsed(copy.deepcopy(r),copy.deepcopy(THRESHOLD_ENV)),"admissible":candidate_admissible(copy.deepcopy(r),copy.deepcopy(THRESHOLD_ENV))}

def compare(r):
    c,o=eval_c(r),eval_o(r)
    mismatch=[k for k in ("representable","noncollapsed","admissible") if c[k]!=o[k]]
    return {"row_id":r["row_id"],"parent_id":r.get("parent_id"),"family":r["family"],"candidate":c,"cleanroom":o,"agreement":not mismatch,"mismatch_fields":mismatch,"input_sha256":sha256_bytes(canonical(r).encode())}

PASS1=[compare(r) for r in ROWS]
shuffled=copy.deepcopy(ROWS); random.Random(26012601).shuffle(shuffled)
PASS2=[compare(r) for r in shuffled]
def ndigest(results): return sha256_bytes(canonical(sorted(results,key=lambda x:x["row_id"])).encode())
D1,D2=ndigest(PASS1),ndigest(PASS2)
REPLAY_AGREEMENT=D1==D2
COUNTEREXAMPLES=[x for x in PASS1 if not x["agreement"]]
print("Evaluations:",len(ROWS)*2,"Counterexamples:",len(COUNTEREXAMPLES),"Replay:",REPLAY_AGREEMENT)


In [ ]:
LOOKUP={x["row_id"]:x for x in PASS1}
INVARIANTS=[]
expected={"wrong_type":"REJECT_TYPE","empty_context":"REJECT_CONTEXT","cross_context":"REJECT_CONTEXT","unknown_profile":"REJECT_PROFILE","missing_witness":"REJECT_WITNESS","witness_scalar":"REJECT_WITNESS","witness_corruption":"REJECT_WITNESS","missing_history":"REJECT_HISTORY","empty_history":"REJECT_HISTORY","malformed_history":"REJECT_HISTORY","boolean_history_step":"REJECT_HISTORY","duplicate_history_step":"REJECT_HISTORY","descending_history":"REJECT_HISTORY","false_terminal":"REJECT_HISTORY","combined_type_context_fault":"REJECT_TYPE","combined_context_profile_fault":"REJECT_CONTEXT","combined_all_faults":"REJECT_TYPE"}
for b in BASELINES:
    bid=b["row_id"]
    INVARIANTS.append({"id":"baseline_admission","row_id":bid,"pass":LOOKUP[bid]["cleanroom"]=={"row_id":bid,"representable":"REPRESENTABLE","noncollapsed":"NON_COLLAPSED","admissible":True}})
    for fam,exp in expected.items():
        rid=f"{bid}__{fam}"; INVARIANTS.append({"id":"representability_rejection_precedence","row_id":rid,"expected":exp,"pass":LOOKUP[rid]["cleanroom"]["representable"]==exp})
    for fam in ("threshold_minus_epsilon","threshold_exact"):
        rid=f"{bid}__{fam}"; INVARIANTS.append({"id":"strict_threshold_rejection","row_id":rid,"pass":LOOKUP[rid]["cleanroom"]["noncollapsed"]=="REJECT_SUBTHRESHOLD"})
    rid=f"{bid}__threshold_plus_epsilon"; INVARIANTS.append({"id":"strict_threshold_admission","row_id":rid,"pass":LOOKUP[rid]["cleanroom"]["noncollapsed"]=="NON_COLLAPSED"})
    rid=f"{bid}__metadata_enrichment"; INVARIANTS.append({"id":"metadata_enrichment_invariance","row_id":rid,"pass":all(LOOKUP[rid]["cleanroom"][k]==LOOKUP[bid]["cleanroom"][k] for k in ("representable","noncollapsed","admissible"))})
INVARIANT_FAILURES=[x for x in INVARIANTS if not x["pass"]]
print("Invariant checks:",len(INVARIANTS),"Failures:",len(INVARIANT_FAILURES))


In [ ]:
if COUNTEREXAMPLES or INVARIANT_FAILURES or not REPLAY_AGREEMENT:
    OUTCOME="FAIL_COUNTEREXAMPLE_OR_INVARIANT_FAILURE"
elif USE_EXTERNAL_CANDIDATE:
    OUTCOME="PASS_BOUNDED_COMPARISON_EXTERNAL_CANDIDATE"
else:
    OUTCOME="PASS_HARNESS_VALIDATION_REFERENCE_STANDIN"
EVIDENCE_CEILING="C2_LIMITATION_OR_NEGATIVE_RESULT" if USE_EXTERNAL_CANDIDATE else "C1_EXECUTION_RECONSTRUCTION"
summary={"notebook_id":NOTEBOOK_ID,"result_id":RESULT_ID,"generated_at":datetime.now(timezone.utc).isoformat(),"run_mode":RUN_MODE,"candidate_id":CANDIDATE_ID,"external_candidate_enabled":USE_EXTERNAL_CANDIDATE,"evidence_ceiling":EVIDENCE_CEILING,"counts":{"baselines":len(BASELINES),"fault_families":len(FAULTS),"rows_per_pass":len(ROWS),"total_evaluations":len(ROWS)*2,"counterexamples":len(COUNTEREXAMPLES),"invariant_checks":len(INVARIANTS),"invariant_failures":len(INVARIANT_FAILURES)},"replay":{"pass1_digest":D1,"pass2_digest":D2,"agreement":REPLAY_AGREEMENT},"corpus_sha256":CORPUS_SHA256,"outcome":OUTCOME,"blocked_interpretations":["Universal equivalence","Formal proof","Theorem promotion","Authoritative governed comparison while REFERENCE_STANDIN is active"]}
write_jsonl(OUT/"candidate_outputs.jsonl",[eval_c(r) for r in ROWS])
write_jsonl(OUT/"cleanroom_outputs.jsonl",[eval_o(r) for r in ROWS])
write_jsonl(OUT/"comparison_rows.jsonl",PASS1)
write_jsonl(OUT/"comparison_rows_replay_shuffled.jsonl",PASS2)
write_jsonl(OUT/"invariant_checks.jsonl",INVARIANTS)
write_json(OUT/"comparison_summary.json",summary)
write_json(OUT/"counterexamples.json",{"result_id":RESULT_ID,"counterexamples":COUNTEREXAMPLES,"invariant_failures":INVARIANT_FAILURES,"replay_agreement":REPLAY_AGREEMENT,"preservation_rule":"Every disagreement and invariant failure is retained."})
summary


In [ ]:
files=sorted(p for p in OUT.iterdir() if p.is_file() and p.name not in {"manifest.json",f"{RESULT_ID}.zip"})
manifest={"notebook_id":NOTEBOOK_ID,"result_id":RESULT_ID,"created_at":datetime.now(timezone.utc).isoformat(),"run_mode":RUN_MODE,"candidate_id":CANDIDATE_ID,"evidence_ceiling":EVIDENCE_CEILING,"runtime":{"python":sys.version,"platform":platform.platform()},"artifacts":[{"name":p.name,"bytes":p.stat().st_size,"sha256":sha256_file(p)} for p in files],"outcome":OUTCOME}
write_json(OUT/"manifest.json",manifest)
archive=OUT/f"{RESULT_ID}.zip"
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p!=archive: z.write(p,arcname=p.name)
print("Outcome:",OUTCOME)
print("Evidence ceiling:",EVIDENCE_CEILING)
print("Archive:",archive.resolve())
print("Archive SHA-256:",sha256_file(archive))


In [ ]:
archive_path=OUT/f"{RESULT_ID}.zip"
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("Result archive:",archive_path.resolve())
